# T1 — Workflows & Human-in-the-Loop · Jour 3

Au **Jour 2 (Atelier 9)**, vous avez découvert le HITL « haut niveau » avec
`create_agent` + `HumanInTheLoopMiddleware`. Ici on descend d'un cran : on construit le
**graphe d'état à la main** (`StateGraph` + `interrupt()`), ce qui donne le contrôle
total nécessaire à un **workflow de production** — et on livre la promesse laissée en
suspens en J2 : un **checkpointer persistant** qui survit à l'arrêt du processus.

## 🎯 Objectifs

À la fin de ce notebook, vous savez :

1. suspendre un graphe avec **`interrupt()`** et le reprendre avec **`Command(resume=...)`** ;
2. router **après** la décision humaine (approuver / éditer / rejeter) avec **`Command(goto=...)`** ;
3. remplacer `InMemorySaver` par un **checkpointer durable** (`SqliteSaver`) et vérifier
   que l'état **survit à un redémarrage** ;
4. assembler un **agent email de support** réaliste où seuls les cas sensibles passent
   par une **validation humaine** ;
5. tenir un **journal d'audit** des interventions et rejouer l'historique d'un thread.

## 🗺️ Plan

| Partie | Sujet |
|---|---|
| 1 | Rappel : `interrupt()` minimal + reprise |
| 2 | Router après décision : `Command(goto=...)` (approve/edit/reject) |
| 3 | Checkpointer **durable** : `SqliteSaver` survit au redémarrage |
| 4 | Capstone : l'**agent email** de support avec HITL sélectif |
| 5 | Journal d'audit & rejeu de l'historique (`get_state_history`) |

## 🧩 Prérequis

- **J2 · Atelier 9** (HITL) et **J1 · T2 Workflows** (StateGraph, nœuds, arêtes).
- Notions supposées acquises : `StateGraph`, nœud, arête, `thread_id`, `interrupt`.

## 🧭 Agent ou workflow ? (la distinction qui structure tout le Jour 3)

Avant d'écrire une ligne de code, il faut trancher une question d'architecture. LangChain
la pose noir sur blanc :

> « **Workflows have predetermined code paths** and are designed to operate in a certain order.
> **Agents are dynamic and define their own processes and tool usage**. »
> — [LangGraph · Workflows & Agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)

Autrement dit :

- un **workflow** suit un **chemin décidé par *vous*, le développeur** : la séquence, les
  branches, les boucles sont **fixées à l'avance** dans le graphe ;
- un **agent** décide **lui-même**, à l'exécution, quelles actions et quels outils employer,
  dans une **boucle** qui continue tant que la tâche n'est pas résolue.

### Le compromis central

| | **Workflow** | **Agent** |
|---|---|---|
| Qui choisit le chemin ? | le développeur (à l'avance) | le LLM (à l'exécution) |
| Ce qu'on gagne | **prévisibilité, contrôle, traçabilité** | **flexibilité, autonomie** |
| Débogage | chaque étape est visible et reproductible | trajectoire variable d'un run à l'autre |
| Idéal quand… | le process est **connu et stable** | « **problems and solutions are unpredictable** » |

Ce n'est pas un choix idéologique : on prend un **workflow** dès qu'un métier exige de
**garantir l'ordre des étapes**, de **rejouer** après panne et de **placer un contrôle
humain** à un endroit précis — exactement le sujet de ce thème.

### Ils ne s'opposent pas : ils se **composent**

Le point clé, souvent manqué : un **nœud** de workflow peut *être* n'importe quoi.

> « Each node in your workflow can be **a simple function, an LLM call, or an entire agent
> with tools**. »
> — [LangChain · Custom workflow](https://docs.langchain.com/oss/python/langchain/multi-agent/custom-workflow)

L'exemple officiel du **RAG personnalisé** illustre les trois natures de nœud côte à côte :

| Nœud | Nature | Rôle |
|---|---|---|
| `rewrite` | **appel LLM** | reformule la question (sortie structurée) |
| `retrieve` | **déterministe** | recherche vectorielle — *aucun LLM* |
| `agent` | **agent** | raisonne sur le contexte et **décide** d'appeler un outil |

Le **workflow orchestre**, l'**agent décide** — chacun là où il est le plus solide.

> **Note honnête.** Par convention, la Partie 4 s'appelle
> « **agent email** ». Structurellement, c'en est pourtant un **workflow** : le chemin
> *classer → chercher → rédiger → (revue) → envoyer* est **fixe**. Le LLM y prend des
> décisions **locales** (la classification, le contenu), mais **ne choisit pas le graphe**.
> C'est précisément ce qui le rend **auditable** et compatible avec un point **HITL** — au
> contraire de l'agent « libre » de J2·A9, construit avec `create_agent`.

## 📖 Glossaire express

- **`StateGraph`** — graphe dont les nœuds lisent/écrivent un **état** partagé (un `TypedDict`).
- **Reducer** — règle de fusion d'un champ d'état (ex. `operator.add` **concatène** les listes).
- **`interrupt(payload)`** — suspend le graphe et **renvoie `payload`** à l'appelant (la question posée à l'humain).
- **`Command(resume=valeur)`** — **reprend** le graphe en injectant la réponse humaine.
- **`Command(goto=..., update=...)`** — depuis un nœud, **route** vers un autre nœud et met à jour l'état.
- **Checkpointer** — persistance de l'état par `thread_id` ; `InMemorySaver` (RAM) vs `SqliteSaver` (disque).
- **`thread_id`** — identifiant d'une exécution ; **reprendre = réutiliser le même**.
- **`get_state_history`** — relit tous les checkpoints d'un thread (audit / *time-travel*).

## 🛠️ Dépendances

`langgraph` et `langchain-mistralai` viennent du Jour 2. Ce thème ajoute **un seul** paquet,
pour le checkpointer durable sur disque :

```powershell
uv pip install langgraph-checkpoint-sqlite
```

## Configuration de l'environnement

On lit `MISTRAL_API_KEY` et `MISTRAL_SERVER_URL` depuis l'environnement (ou un `.env`
local) et on normalise l'URL pour qu'elle se termine par exactement un `/v1`.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

_base = os.environ["MISTRAL_SERVER_URL"].rstrip("/")
MISTRAL_ENDPOINT = _base if _base.endswith("/v1") else _base + "/v1"

import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "util"))
from env_utils import doublecheck_env

doublecheck_env("../.env")

In [ ]:
from langchain_mistralai import ChatMistralAI

MODEL = "mistral-medium-latest"

llm = ChatMistralAI(
    model=MODEL,
    temperature=0,
    endpoint=MISTRAL_ENDPOINT,
)

print(f"✅ LLM : {MODEL} · temperature=0")

---
## Partie 1 — Rappel : `interrupt()` minimal

**Pourquoi ?** Avant l'agent complet, isolons le mécanisme. Un graphe compilé **avec un
checkpointer** peut s'**arrêter** au milieu d'un nœud grâce à `interrupt(payload)` :
l'appel `invoke` rend la main et le `payload` décrit **ce qu'on demande à l'humain**.
On reprend plus tard, **sur le même `thread_id`**, avec `Command(resume=réponse)`.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


class MiniState(TypedDict):
    action: str
    resultat: str


def demander_validation(state: MiniState) -> MiniState:
    # interrupt() suspend le graphe et renvoie ce dictionnaire à l'appelant.
    reponse = interrupt({"action": state["action"], "question": "Autoriser ? (oui/non)"})
    if reponse == "oui":
        return {"resultat": f"✅ '{state['action']}' exécutée"}
    return {"resultat": f"🚫 '{state['action']}' annulée"}


mini = StateGraph(MiniState)
mini.add_node("valider", demander_validation)
mini.add_edge(START, "valider")
mini.add_edge("valider", END)

mini_app = mini.compile(checkpointer=InMemorySaver())
print("✅ Graphe minimal compilé")

In [ ]:
config = {"configurable": {"thread_id": "mini-1"}}

# 1) Lancement : le graphe s'arrête sur l'interruption.
resultat = mini_app.invoke({"action": "supprimer_dossier"}, config)

interruption = resultat["__interrupt__"][0]
print("⏸️  Interruption reçue :", interruption.value)

# 2) Décision humaine → reprise sur le MÊME thread_id.
resultat = mini_app.invoke(Command(resume="non"), config)
print("▶️  Après reprise :", resultat["resultat"])

> **À retenir.** Trois ingrédients indissociables : un **checkpointer** (sinon pas de
> reprise possible), le **même `thread_id`** au lancement et à la reprise, et le couple
> **`interrupt` / `Command(resume=...)`**.

---
## Partie 2 — Router après la décision : `Command(goto=...)`

**Pourquoi ?** Un vrai workflow ne fait pas que reprendre : il **bifurque** selon la
décision. Un nœud peut renvoyer un **`Command`** qui, en une fois, **met à jour l'état**
(`update=`) *et* **choisit le nœud suivant** (`goto=`). On modélise ici une revue de
document à trois issues : **approuver → finaliser**, **éditer → finaliser**, **rejeter → archiver**.

On en profite pour introduire le **journal d'audit** : le champ `audit` utilise le reducer
`operator.add`, donc chaque nœud qui écrit une entrée l'**ajoute** au lieu d'écraser.

In [ ]:
import operator
from typing import Annotated
from datetime import datetime, timezone


class RevueState(TypedDict):
    document: str
    verdict: str
    audit: Annotated[list[dict], operator.add]  # reducer : concatène les entrées


def _entree_audit(decision: str, doc: str) -> dict:
    return {
        "decision": decision,
        "horodatage": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "extrait": doc[:40],
    }


def revue_humaine(state: RevueState) -> Command:
    decision = interrupt({
        "document": state["document"],
        "action": "Relire : approuver / editer / rejeter",
    })
    # decision est un dict fourni par l'humain, ex. {"choix": "editer", "texte": "..."}
    choix = decision.get("choix")

    if choix == "approuver":
        return Command(update={"audit": [_entree_audit("approuvé", state["document"])]},
                       goto="finaliser")
    if choix == "editer":
        nouveau = decision.get("texte", state["document"])
        return Command(update={"document": nouveau,
                               "audit": [_entree_audit("édité", nouveau)]},
                       goto="finaliser")
    # rejet par défaut
    return Command(update={"audit": [_entree_audit("rejeté", state["document"])]},
                   goto="archiver")


def finaliser(state: RevueState) -> RevueState:
    return {"verdict": "publié"}


def archiver(state: RevueState) -> RevueState:
    return {"verdict": "archivé (rejeté)"}


revue = StateGraph(RevueState)
revue.add_node("revue_humaine", revue_humaine)
revue.add_node("finaliser", finaliser)
revue.add_node("archiver", archiver)
revue.add_edge(START, "revue_humaine")
revue.add_edge("finaliser", END)
revue.add_edge("archiver", END)

revue_app = revue.compile(checkpointer=InMemorySaver())
print("✅ Graphe de revue compilé (3 issues : finaliser / archiver)")

In [ ]:
# Scénario : on ÉDITE le document avant de l'approuver.
cfg = {"configurable": {"thread_id": "revue-edit"}}

etat = revue_app.invoke({"document": "Note de service : congés fermés en août."}, cfg)
print("⏸️ ", etat["__interrupt__"][0].value["action"])

decision = {"choix": "editer", "texte": "Note : les congés d'août sont soumis à validation."}
final = revue_app.invoke(Command(resume=decision), cfg)

print("Verdict :", final["verdict"])
print("Document :", final["document"])
print("Audit   :", final["audit"])

> **Piège.** Après `interrupt()`, la reprise **ré-exécute le nœud depuis le début** — la
> ligne `interrupt(...)` renvoie alors la valeur de `resume`, et le code **en dessous**
> s'exécute pour de vrai. N'y placez donc aucun **effet de bord** (envoi d'email, écriture
> en base) *avant* l'`interrupt`, sinon il se produira **deux fois**.

---
## Partie 3 — Checkpointer **durable** : `SqliteSaver`

**Pourquoi ?** `InMemorySaver` garde l'état en **RAM** : si le kernel (ou le serveur) redémarre
pendant l'attente d'une décision humaine — qui peut prendre des heures — **tout est perdu**.
Un checkpointer **durable** écrit chaque checkpoint sur **disque**. On reprend même après
un crash, tant qu'on réutilise le **même `thread_id`**.

On le prouve en simulant **deux processus** : le premier s'arrête sur l'interruption puis
**ferme sa connexion** (comme s'il s'arrêtait) ; le second **rouvre le même fichier** et
**reprend** l'exécution — l'état vient donc du **disque**, pas de la mémoire.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

DB = "checkpoints_t1.db"
if os.path.exists(DB):
    os.remove(DB)  # départ propre si le notebook est relancé
cfg_durable = {"configurable": {"thread_id": "revue-durable"}}

# --- « Processus 1 » : démarre, s'arrête sur l'interruption, puis s'éteint ---
conn1 = sqlite3.connect(DB, check_same_thread=False)
app1 = revue.compile(checkpointer=SqliteSaver(conn1))
app1.invoke({"document": "Contrat fournisseur à signer."}, cfg_durable)
print("Processus 1 : interrompu, état écrit sur disque.")
conn1.close()  # simule l'arrêt du processus (RAM perdue)

In [ ]:
# --- « Processus 2 » : nouvelle connexion sur le MÊME fichier, reprend ---
conn2 = sqlite3.connect(DB, check_same_thread=False)
app2 = revue.compile(checkpointer=SqliteSaver(conn2))

# L'état est reconstruit depuis le disque, pas depuis la mémoire.
etat_repris = app2.get_state(cfg_durable)
print("Nœud en attente :", etat_repris.next)  # ('revue_humaine',)

final = app2.invoke(Command(resume={"choix": "approuver"}), cfg_durable)
print("Verdict après reprise :", final["verdict"])
print("Audit :", final["audit"])
conn2.close()

> **À retenir.** `InMemorySaver` → **développement** ; `SqliteSaver` → mono-machine durable ;
> **Postgres/Redis** → production distribuée. On ne change **que la ligne du checkpointer** :
> la logique du graphe reste identique. C'est l'intérêt de séparer *orchestration* et *persistance*.

---
## Partie 4 — Capstone : l'agent email de support

**Pourquoi ?** On assemble tout dans un cas réaliste. Un email de support arrive ; l'agent
le **classe**, cherche en **parallèle** dans la doc et le suivi de bugs, **rédige** une
réponse, puis — c'est le cœur du HITL — **n'interrompt que si l'enjeu est fort**
(urgence *high/critical* ou intention *complexe*). Les cas simples partent **sans humain**.

### Topologie

```
START → read_email → classify_intent ─┬→ search_documentation ─┐
                                       └→ bug_tracking ─────────┴→ write_response
                                                                      │ (Command goto)
                                        ┌─────────────────────────────┤
                                        ▼                             ▼
                                   human_review  ── approuvé ──►  send_reply → END
                                        │
                                     rejeté → END
```

On classe avec **la sortie structurée de Mistral** (`with_structured_output`, vue en J2·A7),
qui s'appuie sur le *function calling* du modèle.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field


class EmailClassification(BaseModel):
    """Schéma que le modèle Mistral doit remplir pour classer l'email."""
    intent: Literal["question", "bug", "facturation", "fonctionnalite", "complexe"] = Field(
        description="Intention principale de l'email"
    )
    urgency: Literal["basse", "moyenne", "haute", "critique"] = Field(
        description="Niveau d'urgence"
    )
    topic: str = Field(description="Sujet en quelques mots")
    summary: str = Field(description="Résumé en une phrase")


class EmailState(TypedDict):
    # Entrée
    email_content: str
    sender_email: str
    email_id: str
    # Enrichissements
    classification: Optional[dict]
    ticket_id: Optional[str]
    search_results: Optional[list]
    # Sortie
    draft_response: Optional[str]
    sent: Optional[bool]
    # Traçabilité (reducer : concatène)
    audit: Annotated[list[dict], operator.add]


print("✅ Schémas définis")

In [ ]:
import uuid

structured_llm = llm.with_structured_output(EmailClassification)


def read_email(state: EmailState) -> EmailState:
    # Ici, un vrai connecteur (IMAP/API) parserait l'email. On passe l'entrée telle quelle.
    return {}


def classify_intent(state: EmailState) -> EmailState:
    prompt = (
        "Classe cet email de support client.\n\n"
        f"Expéditeur : {state['sender_email']}\n"
        f"Email : {state['email_content']}"
    )
    classification = structured_llm.invoke(prompt)
    # On stocke un dict (sérialisable par le checkpointer).
    return {"classification": classification.model_dump()}


def search_documentation(state: EmailState) -> EmailState:
    # Simulation d'une recherche RAG dans la base de connaissances (cf. T2).
    sujet = (state.get("classification") or {}).get("topic", "")
    return {"search_results": [f"[doc] Article pertinent sur : {sujet}"]}


def bug_tracking(state: EmailState) -> EmailState:
    # N'ouvre un ticket que pour les bugs.
    if (state.get("classification") or {}).get("intent") == "bug":
        return {"ticket_id": f"BUG-{uuid.uuid4().hex[:8]}"}
    return {}


def write_response(state: EmailState) -> Command[Literal["human_review", "send_reply"]]:
    c = state.get("classification") or {}
    contexte = "\n".join(state.get("search_results") or [])
    ticket = f"\nTicket ouvert : {state['ticket_id']}" if state.get("ticket_id") else ""

    prompt = (
        "Rédige une réponse de support professionnelle, en français, à cet email.\n\n"
        f"Email : {state['email_content']}\n"
        f"Résumé : {c.get('summary')}\n"
        f"Contexte documentaire :\n{contexte}{ticket}"
    )
    reponse = llm.invoke(prompt)

    # HITL sélectif : on n'interrompt que si l'enjeu est fort.
    besoin_revue = c.get("urgency") in ("haute", "critique") or c.get("intent") == "complexe"
    return Command(
        update={"draft_response": reponse.content},
        goto="human_review" if besoin_revue else "send_reply",
    )


def human_review(state: EmailState) -> Command[Literal["send_reply"]]:
    c = state.get("classification") or {}
    decision = interrupt({
        "email_id": state["email_id"],
        "urgency": c.get("urgency"),
        "intent": c.get("intent"),
        "draft_response": state.get("draft_response", ""),
        "action": "Relire puis approuver (avec édition possible) ou rejeter",
    })

    entree = {
        "email_id": state["email_id"],
        "decision": "approuvé" if decision.get("approved") else "rejeté",
        "horodatage": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    if decision.get("approved"):
        return Command(
            update={
                "draft_response": decision.get("edited_response") or state.get("draft_response", ""),
                "audit": [entree],
            },
            goto="send_reply",
        )
    return Command(update={"audit": [entree], "sent": False}, goto=END)


def send_reply(state: EmailState) -> EmailState:
    # Ici, un vrai connecteur enverrait l'email. On journalise l'envoi.
    print(f"📤 Réponse envoyée à {state['sender_email']} (email {state['email_id']})")
    return {"sent": True}


print("✅ Nœuds définis")

In [ ]:
builder = StateGraph(EmailState)
builder.add_node("read_email", read_email)
builder.add_node("classify_intent", classify_intent)
builder.add_node("search_documentation", search_documentation)
builder.add_node("bug_tracking", bug_tracking)
builder.add_node("write_response", write_response)
builder.add_node("human_review", human_review)
builder.add_node("send_reply", send_reply)

builder.add_edge(START, "read_email")
builder.add_edge("read_email", "classify_intent")
# Fan-out : doc + suivi de bugs en parallèle...
builder.add_edge("classify_intent", "search_documentation")
builder.add_edge("classify_intent", "bug_tracking")
# ...puis fan-in sur la rédaction.
builder.add_edge("search_documentation", "write_response")
builder.add_edge("bug_tracking", "write_response")
builder.add_edge("send_reply", END)
# write_response et human_review routent via Command(goto=...), pas d'arête statique.

if os.path.exists("emails_t1.db"):
    os.remove("emails_t1.db")  # départ propre si le notebook est relancé
conn_email = sqlite3.connect("emails_t1.db", check_same_thread=False)
email_app = builder.compile(checkpointer=SqliteSaver(conn_email))

print(email_app.get_graph().draw_mermaid())

### Scénario 1 — email simple → **envoi automatique**

Une question de faible urgence ne mobilise **personne** : l'agent répond seul.

In [ ]:
email_simple = {
    "email_content": "Bonjour, comment réinitialiser mon mot de passe ? Merci.",
    "sender_email": "client@exemple.fr",
    "email_id": "MAIL-1001",
    "audit": [],
}
cfg1 = {"configurable": {"thread_id": "mail-1001"}}

res1 = email_app.invoke(email_simple, cfg1)
print("\nClassification :", res1["classification"])
print("Interruption ? :", "__interrupt__" in res1)
print("Envoyé ? :", res1.get("sent"))

### Scénario 2 — email critique → **validation humaine**

Un incident critique déclenche `interrupt()` : l'agent prépare un brouillon mais **attend**.

In [ ]:
email_critique = {
    "email_content": (
        "URGENT : votre service est indisponible depuis 2 heures, "
        "nous perdons des ventes. C'est inacceptable, réagissez immédiatement !"
    ),
    "sender_email": "grand.compte@exemple.fr",
    "email_id": "MAIL-2002",
    "audit": [],
}
cfg2 = {"configurable": {"thread_id": "mail-2002"}}

res2 = email_app.invoke(email_critique, cfg2)

if "__interrupt__" in res2:
    demande = res2["__interrupt__"][0].value
    print("⏸️  Validation demandée (urgence =", demande["urgency"], ")\n")
    print("Brouillon proposé :\n", demande["draft_response"][:500])
else:
    print("Envoyé sans revue :", res2.get("sent"))

In [ ]:
# Décision humaine : on approuve en éditant légèrement le brouillon.
brouillon = res2["__interrupt__"][0].value["draft_response"]
decision = {
    "approved": True,
    "edited_response": brouillon + "\n\nNous vous tenons informé toutes les 30 minutes.",
}

final2 = email_app.invoke(Command(resume=decision), cfg2)
print("Envoyé ? :", final2.get("sent"))
print("Audit :", final2["audit"])

> **Note honnête.** En atelier, la décision humaine est un **dictionnaire codé en dur** ;
> dans une vraie application, ce dictionnaire viendrait d'une **interface** (bouton
> Approuver/Rejeter, zone d'édition). Le graphe, lui, ne change pas : il attend simplement
> une valeur de `resume`.

---
## Partie 5 — Journal d'audit & rejeu de l'historique

**Pourquoi ?** Un workflow qui touche des clients doit être **auditable** : qui a décidé
quoi, et quand. On a déjà accumulé un champ `audit`. Le checkpointer offre en plus le
**rejeu complet** de tous les états traversés par un thread, via `get_state_history` —
utile pour le débogage et la conformité.

In [ ]:
# 1) Le journal d'audit métier (nos entrées explicites).
print("=== Journal d'audit (MAIL-2002) ===")
for entree in final2["audit"]:
    print(f"- {entree['horodatage']} · {entree['decision']} · {entree['email_id']}")

# 2) L'historique technique des checkpoints (time-travel).
print("\n=== Checkpoints traversés (du plus récent au plus ancien) ===")
for snap in email_app.get_state_history(cfg2):
    prochaine = snap.next or ("END",)
    print(f"- prochaine étape : {prochaine}")

> **À retenir.** Deux niveaux de traçabilité complémentaires : l'**audit métier** (ce que
> *vous* journalisez dans l'état, lisible par un humain) et l'**historique des checkpoints**
> (fourni *gratuitement* par le checkpointer, pour rejouer ou reprendre à n'importe quel point).

---
## 🧭 Récapitulatif

| Besoin | Outil LangGraph |
|---|---|
| Suspendre pour un humain | `interrupt(payload)` |
| Reprendre | `Command(resume=valeur)` sur le **même** `thread_id` |
| Router après décision | `Command(goto=..., update=...)` |
| Survivre à un redémarrage | checkpointer durable (`SqliteSaver`, Postgres…) |
| HITL sélectif | interrompre **seulement** si l'enjeu est fort |
| Auditer | champ `audit` (reducer) + `get_state_history` |

## 🔭 Pour aller plus loin

- **Éditer l'état arbitrairement** avant reprise : `app.update_state(config, {...})`.
- **Reprendre à un checkpoint antérieur** (*time-travel*) : rejouer depuis un `checkpoint_id`.
- **Postgres** pour la production distribuée : `langgraph-checkpoint-postgres`.
- Brancher le **RAG (T2)** dans `search_documentation` pour des réponses réellement ancrées.

## 📚 Ressources

- [LangGraph — Workflows & Agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)
- [LangGraph — Human-in-the-loop](https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/)
- [LangGraph — Persistence & checkpointers](https://langchain-ai.github.io/langgraph/concepts/persistence/)
- [LangGraph — `interrupt` API](https://langchain-ai.github.io/langgraph/reference/types/#langgraph.types.interrupt)
- [Mistral — Function calling](https://docs.mistral.ai/capabilities/function_calling/)